# Curve Bootstrapping, Swap Pricing & Sensitivity Analysis

This notebook:
1. Bootstraps **SOFR**, **TermSOFR3m**, **ICP**, and **CLP/USD collateral** curves
2. Prices a **5Y SOFR IRS**, a **TermSOFR3m basis swap**, and a **CLP ICP OIS swap**
3. Computes the **CLP/USD cross-currency basis** from the collateral curve
4. Computes **DV01 / sensitivity** per pillar by bumping exposed `SimpleQuote` objects

In [21]:
import json
from curveengine import CurveEngine
import ORE as ore
import pandas as pd

## Load Configuration & Bootstrap All Curves

Market data loaded from `sensitivity_config.json`.

In [22]:
with open("sensitivity_config.json") as f:
    config = json.load(f)

engine = CurveEngine(config)

# Retrieve curves and indexes
curve_names = ["SOFR", "TermSOFR3m", "ICP", "Collateral(CLP/USD)"]
curves = {}
for name in curve_names:
    curves[name] = {
        'curve': engine.getCurve(name),
        'index': engine.getIndex(name),
        'quotes': engine.getQuotes(name)
    }

refDate = ore.Date(11, 11, 2025)
ore.Settings.instance().evaluationDate = refDate
print(f"Curves built   : SOFR, TermSOFR3m, ICP, Collateral(CLP/USD)")

Curves built   : SOFR, TermSOFR3m, ICP, Collateral(CLP/USD)


In [23]:
with open('qs_results.json') as file:
    qs_results = json.load(file)

In [25]:
ore_data = {}
dc = ore.Actual360()
for key,vals in curves.items():
    curve = vals['curve']
    nodes = curve.nodes()
    ore_data[name] = {
        'date': [d.ISO() for d,_ in nodes],
        'yfs':[dc.yearFraction(refDate, d) for d,_ in nodes],
        'dfs': [d for _,d in nodes]
    }

qs_data = {}
dc = ore.Actual360()
for curve in qs_results['curves']:
    nodes = curve['nodes']
    qs_data[curve['name'] ] = {
        'date': [d['date'] for d in nodes],
        'yfs':[d['year_fraction'] for d in nodes],
        'dfs': [d['discount_factor'] for d in nodes],
    }

compare = {}
for k in ore_data.keys():
    ore_df = pd.DataFrame(ore_data[k])
    qs_df = pd.DataFrame(qs_data[k])
    compare[k] = ore_df.merge(qs_df, right_on='date', left_on='date', how='outer', suffixes=['_ore','_qs'])
    compare[k]['diff'] = compare[k]['dfs_ore'] - compare[k]['dfs_qs']

In [ ]:
for k in compare.keys():
    print(f"Curve: {k}")
    display(compare[k])

Curve: SOFR


,date,yfs_ore,dfs_ore,yfs_qs,dfs_qs,diff
0,2025-11-11,0.000000,1.000000,0.000000,1.000000,0.000000e+00
1,2025-11-12,0.002778,0.999880,0.002778,0.999880,1.110223e-16
2,2026-02-11,0.255556,0.989043,0.255556,0.989043,1.110223e-16
3,2026-05-11,0.502778,0.979079,0.502778,0.979079,-6.772360e-15
4,2026-11-11,1.013889,0.959706,1.013889,0.959706,-7.538414e-14
5,2027-11-11,2.027778,0.924388,2.027778,0.924388,1.998401e-15
6,2028-11-11,3.044444,0.890829,3.044444,0.890829,-4.729550e-14
7,2029-11-11,4.058333,0.858579,4.058333,0.858579,2.553513e-15
8,2030-11-11,5.072222,0.827320,5.072222,0.827320,3.663736e-15
9,2032-11-11,7.102778,0.767340,7.102778,0.767340,3.663736e-15


Curve: TermSOFR3m


,date,yfs_ore,dfs_ore,yfs_qs,dfs_qs,diff
0,2025-11-11,0.000000,1.000000,0.000000,1.000000,0.000000e+00
1,2026-02-11,0.255556,0.988968,0.255556,0.988968,1.110223e-16
2,2026-05-11,0.502778,0.979017,0.502778,0.979017,-1.876277e-14
3,2026-11-11,1.013889,0.959552,1.013889,0.959552,-2.109424e-15
4,2027-11-11,2.027778,0.924023,2.027778,0.924023,5.551115e-16
5,2028-11-11,3.044444,0.890223,3.044444,0.890223,-4.829470e-14
6,2029-11-11,4.058333,0.857732,4.058333,0.857732,1.332268e-15
7,2030-11-11,5.072222,0.826214,5.072222,0.826214,2.664535e-15
8,2032-11-11,7.102778,0.765696,7.102778,0.765696,1.776357e-15
9,2035-11-11,10.144444,0.681022,10.144444,0.681022,1.507683e-13


Curve: ICP


,date,yfs_ore,dfs_ore,yfs_qs,dfs_qs,diff
0,2025-11-11,0.000000,1.000000,0.000000,1.000000,0.000000
1,2025-11-12,0.002778,0.999861,0.002778,0.999861,0.000000
2,2026-02-11,0.255556,0.987433,0.255556,0.987433,0.000000
3,2026-05-11,0.502778,0.975717,0.502778,0.975793,-0.000076
4,2026-11-11,1.013889,0.952594,1.013889,0.952619,-0.000026
5,2027-11-11,2.027778,0.910187,2.027778,0.910209,-0.000022
6,2028-11-11,3.044444,0.872270,3.044444,0.872286,-0.000016
7,2030-11-11,5.072222,0.802582,5.072222,0.802622,-0.000040
8,2032-11-11,7.102778,0.740418,7.102778,0.740511,-0.000093
9,2035-11-11,10.144444,0.654326,10.144444,0.654431,-0.000105


Curve: Collateral(CLP/USD)


,date,yfs_ore,dfs_ore,yfs_qs,dfs_qs,diff
0,2025-11-11,0.000000,1.000000,0.000000,1.000000,0.000000e+00
1,2025-12-11,0.083333,0.994817,0.083333,0.994817,-2.731149e-14
2,2026-02-11,0.255556,0.983468,0.255556,0.983468,0.000000e+00
3,2026-05-11,0.502778,0.965857,0.502778,0.965857,-6.772360e-15
4,2026-11-11,1.013889,0.950459,1.013889,0.950459,5.873080e-14
5,2027-11-11,2.027778,0.907731,2.027778,0.907731,3.219647e-15
6,2028-11-11,3.044444,0.869967,3.044444,0.869967,9.214851e-15
7,2030-11-11,5.072222,0.796764,5.072222,0.796764,4.773959e-15
8,2032-11-11,7.102778,0.730045,7.102778,0.730045,-5.639933e-14
9,2035-11-11,10.144444,0.645087,10.144444,0.645087,1.587619e-14


In [29]:
calendar   = ore.NullCalendar()
convention = ore.Following
notional   = 10_000_000.0
start      = calendar.advance(refDate, ore.Period("2D"))
maturity   = calendar.advance(start, ore.Period("5Y"))

sofr_curve = curves['SOFR']['curve']
sofr_index = curves['SOFR']['index']
fixed_sched = ore.Schedule(start, maturity, ore.Period(ore.Semiannual),
                           calendar, convention, convention,
                           ore.DateGeneration.Forward, False)
float_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                           calendar, convention, convention,
                           ore.DateGeneration.Forward, False)

sofr_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, notional,
                             fixed_sched, 0.0378, dc,
                             float_sched, sofr_index, 0.0, dc)

sofr_handle = ore.YieldTermStructureHandle(sofr_curve)
sofr_swap.setPricingEngine(ore.DiscountingSwapEngine(sofr_handle))

print(f"SOFR OIS 5Y  |  NPV = {sofr_swap.NPV():,.2f} USD  |  Fair Rate = {sofr_swap.fairRate():.6%}")

# # Cashflows as DataFrame
# rows = []
# for leg_name, leg_cfs, cast_fn in [("Fixed", sofr_swap.fixedLeg(), ore.as_fixed_rate_coupon),
#                                     ("Float", sofr_swap.floatingLeg(), ore.as_floating_rate_coupon)]:
#     for cf in leg_cfs:
#         cpn = cast_fn(cf)
#         rows.append({
#             "Leg": leg_name,
#             "PayDate": str(cf.date()),
#             "AccStart": str(cpn.accrualStartDate()) if cpn else "",
#             "AccEnd": str(cpn.accrualEndDate()) if cpn else "",
#             "YearFrac": round(cpn.accrualPeriod(), 6) if cpn else 0.0,
#             "Amount": round(cf.amount(), 2),
#             "DF": round(sofr_curve.discount(cf.date()), 10),
#         })

# pd.DataFrame(rows)

SOFR OIS 5Y  |  NPV = 342.60 USD  |  Fair Rate = 3.779250%


In [31]:
sofr_tenors  = ["1D","3M","6M","1Y","2Y","3Y","4Y","5Y","7Y","10Y","15Y","20Y","30Y"]
sofr_quotes  = curves['SOFR']['quotes']
bump = 1e-4
base_npv = sofr_swap.NPV()
rows, total = [], 0.0
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = sofr_swap.NPV() - base_npv
    rq.setValue(orig)
    total += dv01
    rows.append({"Tenor": tenor, "Quote": round(orig, 6), "DV01 (USD/bp)": round(dv01, 2)})
pd.DataFrame(rows)

,Tenor,Quote,DV01 (USD/bp)
0,1D,0.04330,2.75
1,3M,0.04335,2.78
2,6M,0.04250,0.10
3,1Y,0.04100,0.00
4,2Y,0.03920,-0.00
5,3Y,0.03840,-0.00
6,4Y,0.03800,-0.00
7,5Y,0.03780,-4554.84
8,7Y,0.03770,-17.69
9,10Y,0.03790,0.00


In [32]:
ts3m_index = curves['TermSOFR3m']['index']
ts3m_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, notional,
                             fixed_sched, 0.04, dc,
                             float_sched, ts3m_index, 0.0, dc)
ts3m_swap.setPricingEngine(ore.DiscountingSwapEngine(sofr_handle))

print(f"TermSOFR3m 5Y  |  NPV = {ts3m_swap.NPV():,.2f} USD  |  Fair Rate = {ts3m_swap.fairRate():.6%}")

TermSOFR3m 5Y  |  NPV = 88,754.68 USD  |  Fair Rate = 3.805675%


In [36]:
ts3m_tenors  = ["3M","6M","1Y","2Y","3Y","4Y","5Y","7Y","10Y","15Y","20Y","30Y"]
ts3m_quotes  = curves['TermSOFR3m']['quotes']
base_npv_ts  = ts3m_swap.NPV()

rows, total = [], 0.0

# Bump TermSOFR3m (forecast curve)
for tenor, q in zip(ts3m_tenors, ts3m_quotes):
    rq = q.get('spread', q.get('rate'))
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = ts3m_swap.NPV() - base_npv_ts
    rq.setValue(orig)
    total += dv01
    rows.append({"Curve": "TermSOFR3m", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (discount curve)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = ts3m_swap.NPV() - base_npv_ts
    rq.setValue(orig)
    total += dv01
    rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

pd.DataFrame(rows)

,Curve,Tenor,DV01 (USD/bp)
0,TermSOFR3m,3M,5.56
1,TermSOFR3m,6M,-0.01
2,TermSOFR3m,1Y,0.00
3,TermSOFR3m,2Y,-0.00
4,TermSOFR3m,3Y,0.00
5,TermSOFR3m,4Y,0.01
6,TermSOFR3m,5Y,-4576.88
7,TermSOFR3m,7Y,-17.69
8,TermSOFR3m,10Y,-0.00
9,TermSOFR3m,15Y,-0.00


In [34]:
clp_notional = 5_000_000_000.0
clp_fixed_rate = 0.0440

icp_index = curves['ICP']['index']
coll_curve = curves['Collateral(CLP/USD)']['curve']
clp_fixed_sched = ore.Schedule(start, maturity, ore.Period(ore.Semiannual),
                               calendar, convention, convention,
                               ore.DateGeneration.Forward, False)
clp_float_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                               calendar, convention, convention,
                               ore.DateGeneration.Forward, False)

icp_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, clp_notional,
                            clp_fixed_sched, clp_fixed_rate, dc,
                            clp_float_sched, icp_index, 0.0, dc)

# Discount with CLP/USD collateral curve
coll_handle = ore.YieldTermStructureHandle(coll_curve)
icp_swap.setPricingEngine(ore.DiscountingSwapEngine(coll_handle))

print(f"CLP ICP OIS 5Y  |  NPV = {icp_swap.NPV():,.2f} CLP")

CLP ICP OIS 5Y  |  NPV = 40,241.56 CLP


In [38]:
fx_spot = 935.0
icp_tenors  = ["1D","3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y"]
icp_quotes  = curves['ICP']['quotes']
coll_tenors = ["1M","3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y"]
coll_quotes = curves['Collateral(CLP/USD)']['quotes']

base_npv_icp = icp_swap.NPV()
rows, total = [], 0.0

# Bump ICP (forecast curve)
for tenor, q in zip(icp_tenors, icp_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = (icp_swap.NPV() - base_npv_icp) / fx_spot
    rq.setValue(orig)
    total += dv01
    rows.append({"Curve": "ICP", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump CLP_COLLUSD (collateral / discount curve)
for tenor, q in zip(coll_tenors, coll_quotes):
    # FxSwap helpers have fxPoints; Xccy helpers have rate
    if 'fxPoints' in q:
        rq = q['fxPoints']
    elif 'rate' in q:
        rq = q['rate']
        
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = (icp_swap.NPV() - base_npv_icp) / fx_spot
    rq.setValue(orig)
    total += dv01
    rows.append({"Curve": "CLP_COLLUSD", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (affects collateral curve via xccy helpers)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = (icp_swap.NPV() - base_npv_icp) / fx_spot
    rq.setValue(orig)
    total += dv01
    rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})
pd.DataFrame(rows)

,Curve,Tenor,DV01 (USD/bp)
0,ICP,1D,1.46
1,ICP,3M,-0.94
2,ICP,6M,2.45
3,ICP,1Y,-0.12
4,ICP,2Y,-0.74
5,ICP,3Y,0.50
6,ICP,5Y,-2360.84
7,ICP,7Y,-9.10
8,ICP,10Y,-0.00
9,ICP,20Y,-0.00


In [40]:
fx_spot     = 935.0
usd_not     = 10_000_000.0
clp_not     = usd_not * fx_spot
xccy_spread = 0.0050  # 50 bp on CLP (receive) leg

icp_curve = curves['ICP']['curve']
xccy_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                          calendar, convention, convention,
                          ore.DateGeneration.Forward, False)

# Pay USD SOFR flat, Receive CLP ICP + 50 bp
xccy_swap = ore.CrossCcyBasisSwap(
    usd_not,  ore.USDCurrency(), xccy_sched, sofr_index, 0.0, 1.0,
    clp_not,  ore.CLPCurrency(), xccy_sched, icp_index, xccy_spread, 1.0
)

coll_handle = ore.YieldTermStructureHandle(coll_curve)
xccy_engine = ore.CrossCcySwapEngine(
    ore.USDCurrency(), sofr_handle,
    ore.CLPCurrency(), coll_handle,
    ore.QuoteHandle(ore.SimpleQuote(1/fx_spot))
)
xccy_swap.setPricingEngine(xccy_engine)

print(f"CLP/USD XCCY Basis 5Y  |  NPV = {xccy_swap.NPV():,.2f}")
print(f"  Pay USD leg NPV = {xccy_swap.legNPV(0):,.2f}")
print(f"  Rec CLP leg NPV = {xccy_swap.legNPV(1):,.2f}")
print(f"  Fair Spread on CLP leg = {xccy_swap.fairPaySpread()*10000:.2f} bp")

CLP/USD XCCY Basis 5Y  |  NPV = 158,589.08
  Pay USD leg NPV = 0.00
  Rec CLP leg NPV = 158,589.08
  Fair Spread on CLP leg = 34.56 bp


In [41]:
base_npv_xccy = xccy_swap.NPV()
rows, total = [], 0.0

# Bump ICP (CLP forecast curve)
for tenor, q in zip(icp_tenors, icp_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = xccy_swap.NPV() - base_npv_xccy
    rq.setValue(orig)
    total += dv01
    rows.append({"Curve": "ICP", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump CLP_COLLUSD (collateral / discount for CLP leg)
for tenor, q in zip(coll_tenors, coll_quotes):
    if 'fxPoints' in q:
        rq = q['fxPoints']
    elif 'rate' in q:
        rq = q['rate']
    else:
        continue
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = xccy_swap.NPV() - base_npv_xccy
    rq.setValue(orig)
    total += dv01
    rows.append({"Curve": "CLP_COLLUSD", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (USD discount)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    dv01 = xccy_swap.NPV() - base_npv_xccy
    rq.setValue(orig)
    total += dv01
    rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

pd.DataFrame(rows)

,Curve,Tenor,DV01 (USD/bp)
0,ICP,1D,-2.73
1,ICP,3M,1.76
2,ICP,6M,-4.59
3,ICP,1Y,0.23
4,ICP,2Y,1.39
5,ICP,3Y,-0.94
6,ICP,5Y,4414.77
7,ICP,7Y,17.01
8,ICP,10Y,-0.00
9,ICP,20Y,-0.00
